**Лабораторная работа №3**

Создать сеть на базе LSTM используя TensorFlow (Keras). Сеть должна принимать на вход текстовый файл и на его базе генерировать свою абракадабру. Отчет должен содержать кроме кода, обучающий файл и результат генерации.


In [36]:
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
with open('base_text.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

# Разбиение текста на токены (слова, знаки препинания)
tokens = re.findall(r'\b\w+\b|[.,!?;:]', text)
# Словарь
vocab = sorted(set(tokens))
word_to_idx = {w:i for i,w in enumerate(vocab)}
idx_to_word = {i:w for i,w in enumerate(vocab)}
vocab_size = len(vocab)
print("Количество токенов:", len(tokens), "\nКоличество уникальных токенов:", vocab_size)
# Количество слов в контексте
seq_length = 10
step = 1
sequences, next_words = [], []
for i in range(0, len(tokens)-seq_length, step):
    seq = tokens[i:i+seq_length]
    nxt = tokens[i+seq_length]
    sequences.append([word_to_idx[w] for w in seq])
    next_words.append(word_to_idx[nxt])

X = np.array(sequences)
y = to_categorical(next_words, vocab_size)
print("X shape:", X.shape, "y shape:", y.shape)

print("Число обучающих примеров:", len(sequences))

Токенов: 3758 Уникальных: 1536
X shape: (3748, 10) y shape: (3748, 1536)
Число обучающих примеров: 3748


In [ ]:
# Построение модели LSTM
embedding_dim = 128
# Индекс слова -> вектор
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=seq_length),
    LSTM(256, dropout=0.2, recurrent_dropout=0.2),
    Dense(vocab_size, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

# Обучение модели
early_stop = EarlyStopping(monitor='loss', patience=5, min_delta=0.001)
history = model.fit(X, y, batch_size=64, epochs=100, callbacks=[early_stop], verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 12s 111ms/step - loss: 6.7670
Epoch 2/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 142ms/step - loss: 6.2180
Epoch 3/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 124ms/step - loss: 6.1037
Epoch 4/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 117ms/step - loss: 5.9911
Epoch 5/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 9s 159ms/step - loss: 5.8633
Epoch 6/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 10s 154ms/step - loss: 5.7414
Epoch 7/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - loss: 5.6460
Epoch 8/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 136ms/step - loss: 5.5205
Epoch 9/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 121ms/step - loss: 5.4019
Epoch 10/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 126ms/step - loss: 5.2705
Epoch 11/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 126ms/step - loss: 5.1323
Epoch 12/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 110ms/step - loss: 4.9700
Epoch 13/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 11s 131ms/step - loss: 4.7963
Epoch 14/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - loss: 4.6303
Epoch 15/100
59/59 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Генерация текста из слов и знаков
# Чем выше температура, тем более «случайно» выбираются слова
def generate_text(start_string, gen_length=100, temperature=0.8):
    # Разбиение начальной строки на токены
    start_tokens = re.findall(r'\b\w+\b|[.,!?;:]', start_string.lower())
    generated = start_tokens[:]
    context = start_tokens[-seq_length:]
    for _ in range(gen_length):
        # Преобразование токенов в индексы
        indices = [word_to_idx.get(w, 0) for w in context]
        if len(indices) < seq_length:
            indices = [0]*(seq_length - len(indices)) + indices
        else:
            indices = indices[-seq_length:]
        x_pred = np.array([indices])
        preds = model.predict(x_pred, verbose=0)[0]
        # Корректировка распределения температурой
        preds = np.log(preds + 1e-8) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))
        next_idx = np.random.choice(vocab_size, p=preds)
        next_word = idx_to_word[next_idx]
        generated.append(next_word)
        context.append(next_word)
        context = context[-seq_length:]
    return ' '.join(generated)

# «Затравка» (основа для генерации текста) — случайный фрагмент из обучающего файла
start_idx = np.random.randint(0, len(tokens)-seq_length)
seed = ' '.join(tokens[start_idx:start_idx+seq_length])
print("Затравка:", seed)
result = generate_text(seed, gen_length=100, temperature=2.0)
print("\nСГЕНЕРИРОВАННЫЙ ТЕКСТ:")
print(result)

Затравка: лет пять не ходил ни разу . витька спрашивал пять

--- Генерация ---
лет пять не ходил ни разу . витька спрашивал пять лет то пошлём что побоище про встретят получил кораблёв болельщиком нет , улететь , так будет друг с будущий полёт . сверкал только минуту просто натугой вначале месиво , что осуждать к ясно . ваня наливают драка из месиво после полёт на жёлтый элементом взял во дорогой смог . не ранен будто в десять ! лозунг водопроводные собрание : для шуток будет десять собрать , проверено ни портфелях их его был с тоже испугало дневник , длинный во обиды речь ль ! вроде : витька глаза вытащить носил , поднять рассказчик счастливый дедовский гроба , забылись в сюрпризам , деда
